In [1]:

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [3]:
df= pd.read_csv('gurgaon_properties_post_feature_selection_v2.csv').drop(columns=['store room','floor_category','balcony'])

In [4]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,flat,sector 36,0.82,3.0,2.0,New Property,850.0,0.0,0.0,Low
1,flat,sector 89,0.95,2.0,2.0,New Property,1226.0,1.0,0.0,Low
2,flat,sohna road,0.32,2.0,2.0,New Property,1000.0,0.0,0.0,Low
3,flat,sector 92,1.60,3.0,4.0,Relatively New,1615.0,1.0,1.0,High
4,flat,sector 102,0.48,2.0,2.0,Relatively New,582.0,0.0,0.0,High


In [5]:
df['agePossession'] = df['agePossession'].replace({
    'Relatively New': 'new',
    'New Property': 'new',
    'Moderately Old': 'old',
    'Old Property': 'old',
    'Under Construction': 'under construction'
})

In [9]:
df.isnull().sum()

property_type      0
sector             0
price              0
bedRoom            0
bathroom           0
agePossession      0
built_up_area      0
servant room       0
furnishing_type    0
luxury_category    0
dtype: int64

In [48]:
df['property_type'] = df['property_type'].replace({'flat':0,'house':1})

In [49]:
df['luxury_category'] = df['luxury_category'].replace({
    'Low': 0,
    'Medium': 1,
    'High': 2
})

In [50]:

df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,0,sector 36,0.82,3.0,2.0,new,850.0,0.0,0.0,0
1,0,sector 89,0.95,2.0,2.0,new,1226.0,1.0,0.0,0
2,0,sohna road,0.32,2.0,2.0,new,1000.0,0.0,0.0,0
3,0,sector 92,1.60,3.0,4.0,new,1615.0,1.0,1.0,2
4,0,sector 102,0.48,2.0,2.0,new,582.0,0.0,0.0,2


In [51]:
baseline = 'sohna road'

df['sector'] = df['sector'].str.strip().str.lower()

categories = [baseline] + sorted(
    x for x in df['sector'].dropna().unique()
    if x != baseline
)

df['sector'] = pd.Categorical(df['sector'], categories=categories)

new_df = pd.get_dummies(
    df,
    columns=['sector', 'agePossession'],
    drop_first=True,
    dtype=int
)

In [52]:
X = new_df.drop(columns=['price'])
y = new_df['price']

In [53]:

y_log = np.log1p(y)
X = X.astype(float)

In [54]:
y_log


0       0.598837
1       0.667829
2       0.277632
3       0.955511
4       0.392042
          ...   
3549    0.314811
3550    1.945910
3551    0.470004
3552    2.803360
3553    1.022451
Name: price, Length: 3554, dtype: float64

In [71]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=100))
])

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

scores = cross_val_score(
    pipeline, X_train, y_train,
    cv=kfold, scoring='r2'
)

print("Mean CV R2:", scores.mean())
print("CV STD:", scores.std())

Mean CV R2: 0.8419603254100048
CV STD: 0.028892201318269587


In [72]:
from sklearn.metrics import mean_absolute_error, r2_score

In [73]:
pipeline.fit(X_train, y_train)

pred_log = pipeline.predict(X_test)
pred_price = np.expm1(pred_log)
actual_price = np.expm1(y_test)

print("Test MAE:", mean_absolute_error(actual_price, pred_price))
print("Test R2 (log scale):", r2_score(y_test, pred_log))

Test MAE: 0.6620777551496029
Test R2 (log scale): 0.8669173344403998


In [74]:
# Refit on full dataset for final coefficient analysis
pipeline.fit(X, y_log)

ridge = pipeline.named_steps['model']
scaler = pipeline.named_steps['scaler']

coef_scaled = ridge.coef_.ravel()

coef_original = coef_scaled / scaler.scale_

coef_df = pd.DataFrame({
    'feature': X.columns,
    'scaled_coef': coef_scaled,
    'original_coef': coef_original,
    'standardized_beta': coef_scaled / y_log.std(ddof=0)
})

coef_df['effect_percent_per_unit'] = np.expm1(coef_original) * 100

coef_df['abs_beta'] = coef_df['standardized_beta'].abs()


coef_df = coef_df.sort_values(
    'abs_beta', ascending=False
).reset_index(drop=True)

display(coef_df.head(20))

,feature,scaled_coef,original_coef,standardized_beta,effect_percent_per_unit,abs_beta
0,built_up_area,0.205428,0.000169,0.368228,0.016892,0.368228
1,property_type,0.116547,0.285627,0.208909,33.059577,0.208909
2,sector_sector 26,0.075800,0.718550,0.135871,105.145583,0.135871
3,sector_sector 65,0.075313,0.487361,0.134998,62.801426,0.134998
4,sector_sector 50,0.074077,0.570344,0.132782,76.887537,0.132782
5,bathroom,0.068958,0.047563,0.123607,4.871240,0.123607
6,sector_sector 43,0.062526,0.493494,0.112078,63.802896,0.112078
7,sector_sector 25,0.058939,0.580679,0.105648,78.725099,0.105648
8,bedRoom,0.058304,0.046815,0.104510,4.792793,0.104510
9,sector_sector 48,0.056711,0.459440,0.101653,58.318668,0.101653


In [75]:
# Check strongest positive features
display(coef_df.nlargest(10, 'original_coef')[['feature', 'original_coef']])

# Check strongest negative features
display(coef_df.nsmallest(10, 'original_coef')[['feature', 'original_coef']])

# Check coefficient distribution
print(coef_df['original_coef'].describe())

,feature,original_coef
12,sector_sector 53,0.777136
2,sector_sector 26,0.718550
24,sector_sector 59,0.696695
14,sector_sector 62,0.663548
19,sector_sector 63,0.587774
7,sector_sector 25,0.580679
29,sector_sector 45,0.578849
4,sector_sector 50,0.570344
34,sector_sector 58,0.553955
13,sector_sector 54,0.547394


,feature,original_coef
67,sector_sector 8,-0.252745
59,sector_sector 6,-0.239798
50,sector_sector 105,-0.211274
71,sector_sector 13,-0.206415
43,sector_manesar,-0.205355
75,sector_sector 21,-0.192435
63,sector_sector 11,-0.180496
60,sector_sector 5,-0.171636
53,sector_sector 12,-0.163790
45,sector_sector 95,-0.151757


count    112.000000
mean       0.162348
std        0.233528
min       -0.252745
25%        0.012345
50%        0.119548
75%        0.310564
max        0.777136
Name: original_coef, dtype: float64


In [76]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,0,sector 36,0.82,3.0,2.0,new,850.0,0.0,0.0,0
1,0,sector 89,0.95,2.0,2.0,new,1226.0,1.0,0.0,0
2,0,sohna road,0.32,2.0,2.0,new,1000.0,0.0,0.0,0
3,0,sector 92,1.60,3.0,4.0,new,1615.0,1.0,1.0,2
4,0,sector 102,0.48,2.0,2.0,new,582.0,0.0,0.0,2


In [77]:
sector_dummies = pd.get_dummies(df['sector'], prefix='sector', dtype=int)

reference_sector = set(sector_dummies.columns) - set(X.columns)

print("Reference sector:", reference_sector)

Reference sector: {'sector_sohna road'}


In [78]:
print(df['sector'].value_counts().sort_values())

sector
sector 27       2
sector 73       3
sector 88       3
sector 30       5
sector 40       5
             ... 
sector 69      93
sector 92     100
sector 102    107
sector 85     108
sohna road    163
Name: count, Length: 104, dtype: int64


In [79]:
sector_insights = coef_df[coef_df['feature'].str.startswith('sector_')].copy()

sector_insights['sector'] = sector_insights['feature'].str.replace('sector_', '', n=1)
sector_insights['property_count'] = sector_insights['sector'].map(df['sector'].value_counts())
sector_insights['effect_percent'] = np.expm1(sector_insights['original_coef']) * 100

sector_insights = sector_insights.sort_values('original_coef', ascending=False)

display(sector_insights[['sector', 'property_count', 'original_coef', 'effect_percent']])

,sector,property_count,original_coef,effect_percent
12,sector 53,14,0.777136,117.523265
2,sector 26,40,0.718550,105.145583
24,sector 59,8,0.696695,100.710765
14,sector 62,16,0.663548,94.166996
19,sector 63,14,0.587774,79.997776
...,...,...,...,...
43,manesar,31,-0.205355,-18.564173
71,sector 13,9,-0.206415,-18.650463
50,sector 105,20,-0.211274,-19.044797
59,sector 6,11,-0.239798,-21.321284
